# Accuracy Gate — score a trained adapter on held-out puzzles

A standalone evaluator for any adapter produced by the training notebooks in this repo. It loads the base model + your saved LoRA adapter and scores a balanced sample of held-out `train.csv` rows at the official competition sampling params (`temperature=0.6`, `top_p=0.95`, thinking on), and prints per-category accuracy.

Point `ADAPTER_CANDIDATES` (in the *Load the model + adapter* cell) at the folder holding your `adapter_config.json` + `adapter_model.safetensors`. Use this to sanity-check a submission before uploading — a category near 0, or a high `no-box` count, means stop and diagnose.

## 1 · Held-out evaluation rows + answer helpers

Load `train.csv`, define category detection / answer comparison (1% relative tolerance for numbers), and pick the fixed 120 held-out rows (20/category, seed 777).

In [ ]:
import os, re, math, random
import polars as pl

OUTPUT_DIR = "/kaggle/working"
PROMPT_SUFFIX = ("\nPlease put your final answer inside `\\boxed{}`. "
                 "For example: `\\boxed{your answer}`")

TRAIN_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "train.csv",
]
TRAIN_CSV = next((p for p in TRAIN_CANDIDATES if os.path.exists(p)), TRAIN_CANDIDATES[-1])
train = pl.read_csv(TRAIN_CSV)


def categorize(prompt):
    if "bit manipulation" in prompt:   return "bit_manipulation"
    if "gravitational" in prompt:      return "gravity"
    if "unit conversion" in prompt:    return "unit_conversion"
    if "encryption" in prompt:         return "cipher"
    if "numeral system" in prompt:     return "numeral"
    if "equations" in prompt:          return "equation"
    return "unknown"


def extract_answer(text):
    m = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if m:
        ne = [x.strip() for x in m if x.strip()]
        return ne[-1] if ne else m[-1].strip()
    return ""


def compare_answer(stored, predicted):
    stored = str(stored).strip(); predicted = str(predicted).strip()
    if re.fullmatch(r"[01]+", stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored.lower()


_rng = random.Random(777)
EVAL_IDS = set()
for _c in ["numeral", "gravity", "unit_conversion", "cipher", "bit_manipulation", "equation"]:
    _ids = sorted(r["id"] for r in train.iter_rows(named=True) if categorize(r["prompt"]) == _c)
    EVAL_IDS.update(_rng.sample(_ids, 20))
print(f"Held out {len(EVAL_IDS)} evaluation rows (20/category, seed 777).")

## 2 · Load the base model + your trained adapter

Load the bf16 base and attach your saved LoRA adapter via `PeftModel`. Set `ADAPTER_CANDIDATES` to where your adapter files live.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch
import kagglehub
import mamba_ssm
from peft import PeftModel

assert mamba_ssm is not None

ADAPTER_CANDIDATES = [
    "/kaggle/input/nemotron-adapter",
    "/kaggle/working",
    "adapter",
    ".",
]
ADAPTER_DIR = next(
    (p for p in ADAPTER_CANDIDATES if os.path.exists(os.path.join(p, "adapter_config.json"))),
    None,
)
assert ADAPTER_DIR is not None, f"No adapter_config.json found in {ADAPTER_CANDIDATES}"
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")), \
    f"adapter_model.safetensors missing in {ADAPTER_DIR}"
print(f"Adapter: {ADAPTER_DIR}")

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=16384,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    attn_implementation="eager",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
FastLanguageModel.for_inference(model)
model.eval()
tokenizer.padding_side = "left"
print("Base + LoRA adapter loaded.")

## 3 · Run the accuracy gate

Score the held-out rows at the official sampling params and print per-category accuracy. Generation is cache-less (this checkpoint's KV-cache path is unstable) and hard time-boxed.

In [ ]:
# Score the held-out rows at the official eval params (temperature 0.6, top_p 0.95).
# Generation is run cache-less on purpose: this checkpoint's KV-cache path is unstable here.

import gc
import os
import csv
import time

import torch

# ---- Fast FAITHFUL gate: real CoT + official sampling, hard-bounded to <= ~30 min ----
# Cache-less generation is O(n^2) in generated length on this 30B checkpoint (the cache
# crashes — see the docstring below), so a full 120-row x 8192-token gate is hours. We keep
# the measurement faithful (thinking ON, temp 0.6 / top_p 0.95) but bound the cost three
# ways: a token cap, a balanced row subset, and a HARD wall-clock budget that stops early
# and scores whatever completed. Tune these three to trade coverage vs time.
N_PER_CAT_GATE = 10          # rows/category to score (10 -> 60 rows; 20 -> the full 120)
GEN_BATCH = 8                # adaptive: halves to 1 on CUDA OOM
GATE_MAX_NEW_TOKENS = 1024   # real-CoT cap; bounds the n^2 cost AND any one batch's length
GATE_TIME_BUDGET_S = 1300    # HARD stop (~22 min) checked before each batch (+<=1 batch)
GATE_RESULTS_CSV = os.path.join(OUTPUT_DIR, "gate_results.csv")

gc.collect()
torch.cuda.empty_cache()


def _is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


def _gen(prompts, max_new):
    """Cache-less generate with REAL CoT at the official eval params (temp 0.6, top_p 0.95).

    Thinking is ON, so this matches what the host scorer does — it is a faithful accuracy
    measurement (truncated only by GATE_MAX_NEW_TOKENS). Cache-less is mandatory: the
    checkpoint's remote code keeps the cache under a kwarg forward() ignores, so generation
    recomputes the prefix each step. Do NOT add a prepare_inputs shim to 'enable' it — that
    routes the cache into the mamba cuda_kernels_forward path (cache_params.conv_kernel_size
    -> AttributeError) and also makes generate reject attention_mask. Both cost real runs.
    """
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p + PROMPT_SUFFIX}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True,
        )
        for p in prompts
    ]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=max_new,
            do_sample=True, temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[:, enc["input_ids"].shape[1]:]
    decoded = [tokenizer.decode(g, skip_special_tokens=True) for g in gen]
    ntoks = [int((g != tokenizer.pad_token_id).sum()) for g in gen]
    del enc, out, gen
    return decoded, ntoks


def generate_adaptive(prompts, max_new=GATE_MAX_NEW_TOKENS):
    try:
        return _gen(prompts, max_new)
    except Exception as e:
        if not _is_oom(e):
            raise
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            print("    OOM at batch 1 — recording no-box.")
            return [""], [0]
        mid = len(prompts) // 2
        print(f"    OOM at batch {len(prompts)} -> retrying {mid}+{len(prompts) - mid}")
        l_txt, l_tok = generate_adaptive(prompts[:mid], max_new)
        r_txt, r_tok = generate_adaptive(prompts[mid:], max_new)
        return l_txt + r_txt, l_tok + r_tok


def generate_batch(prompts):
    """Back-compat alias used by the smoke-test cell."""
    return generate_adaptive(prompts)


def _gate_rows():
    """Deterministic balanced subset: N_PER_CAT_GATE rows/category from the 120 EVAL_IDS."""
    by_cat = {}
    for r in train.iter_rows(named=True):
        if r["id"] in EVAL_IDS:
            by_cat.setdefault(categorize(r["prompt"]), []).append(r)
    rows = []
    for cat in sorted(by_cat):
        rows.extend(sorted(by_cat[cat], key=lambda r: r["id"])[:N_PER_CAT_GATE])
    return rows


def run_gate(rows):
    """Score rows (real CoT, official params); incremental + hard time-boxed.

    Checks the wall clock before each batch and stops once GATE_TIME_BUDGET_S is exceeded,
    scoring whatever completed. Un-run rows are reported as 'unmeas' (not counted wrong),
    so the gate ALWAYS returns within ~30 min and never silently calls a category bad just
    because it ran out of time. Every finished row is written to GATE_RESULTS_CSV live.
    """
    prompts = [r["prompt"] for r in rows]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    done = [False] * len(prompts)
    outputs = [""] * len(prompts)
    gen_lens = [0] * len(prompts)
    with open(GATE_RESULTS_CSV, "w", newline="") as _f:
        csv.writer(_f).writerow(["id", "category", "answer", "pred", "ok", "ntok"])
    _t0 = time.perf_counter()
    for s in range(0, len(order), GEN_BATCH):
        if time.perf_counter() - _t0 > GATE_TIME_BUDGET_S:
            print(f"  time budget {GATE_TIME_BUDGET_S}s reached — stopping, scoring done rows.")
            break
        idxs = order[s:s + GEN_BATCH]
        try:
            decoded, ntoks = generate_adaptive([prompts[i] for i in idxs])
        except Exception as e:
            print(f"    batch {s} failed ({type(e).__name__}: {e}); recording no-box.")
            torch.cuda.empty_cache()
            decoded, ntoks = [""] * len(idxs), [0] * len(idxs)
        with open(GATE_RESULTS_CSV, "a", newline="") as _f:
            _w = csv.writer(_f)
            for i, txt, nt in zip(idxs, decoded, ntoks):
                outputs[i] = txt
                gen_lens[i] = nt
                done[i] = True
                pred = extract_answer(txt)
                _w.writerow([rows[i]["id"], categorize(rows[i]["prompt"]),
                             rows[i]["answer"], pred,
                             int(compare_answer(rows[i]["answer"], pred)), nt])
        torch.cuda.empty_cache()
        print(f"  {sum(done)}/{len(order)} rows [{time.perf_counter() - _t0:.0f}s]")
    return outputs, gen_lens, done


gate_rows = _gate_rows()
print(f"Gate: {len(gate_rows)} rows ({N_PER_CAT_GATE}/cat), real CoT, "
      f"max_new={GATE_MAX_NEW_TOKENS}, budget {GATE_TIME_BUDGET_S}s.")
outputs, gen_lens, done = run_gate(gate_rows)

stats = {}
for r, txt, nt, d in zip(gate_rows, outputs, gen_lens, done):
    cat = categorize(r["prompt"])
    st = stats.setdefault(cat, dict(n=0, ok=0, nobox=0, toks=0, skip=0))
    if not d:
        st["skip"] += 1
        continue
    pred = extract_answer(txt)
    st["n"] += 1
    st["toks"] += nt
    if not pred:
        st["nobox"] += 1
    if compare_answer(r["answer"], pred):
        st["ok"] += 1

print(f"\n{'category':18s} {'acc':>6} {'done':>5} {'no-box':>7} {'avg-tok':>8} {'unmeas':>7}")
tot_ok = tot_n = 0
for cat in sorted(stats):
    st = stats[cat]
    acc = st["ok"] / st["n"] if st["n"] else float("nan")
    avgt = st["toks"] / st["n"] if st["n"] else 0
    tot_ok += st["ok"]
    tot_n += st["n"]
    print(f"{cat:18s} {acc:>6.2f} {st['n']:>5} {st['nobox']:>7} {avgt:>8.0f} {st['skip']:>7}")
print(f"{'OVERALL':18s} {(tot_ok / tot_n if tot_n else float('nan')):>6.2f} {tot_n:>5}")
print(f"\nResults -> {GATE_RESULTS_CSV}")
print("Read: category ~0 = needs work. High no-box WITH avg-tok near the cap = CoT")
print("truncated (raise GATE_MAX_NEW_TOKENS), not a real failure. 'unmeas' > 0 = time")
print("budget hit (raise GATE_TIME_BUDGET_S or lower N_PER_CAT_GATE).")
